In [1]:
import torch
from PIL import Image
import pandas as pd
from transformers import AutoProcessor, Blip2Processor, Blip2ForImageTextRetrieval,  Blip2ForConditionalGeneration, BitsAndBytesConfig
from tqdm.auto import tqdm

In [2]:
bnb_config = BitsAndBytesConfig(load_in_4bit=True)

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
csv = "powerpoint_data.csv"
df = pd.read_csv(csv)

In [5]:
image_paths = list(df['fixed image dir']) + list(df['waiting image dir']) + list(df['not fixed image dir']) 

In [6]:
processor = Blip2Processor.from_pretrained("Salesforce/blip2-flan-t5-xl")
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-flan-t5-xl"
)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [7]:
model.to(device)
pass

In [8]:
import os
batch_size = 4
generated_texts = []

if not os.path.exists("generated_captions.csv"):
    for i in tqdm(range(0, len(image_paths), batch_size)):
        batch_paths = image_paths[i:i+batch_size]
        images = [Image.open(p) for p in batch_paths]
        
        #inputs = processor(images=images, text=['Focussing on: new, very good, good, acceptable. This is'] * batch_size, return_tensors="pt")
        inputs = processor(images=images, return_tensors="pt")
        inputs = {k: v.to(device, dtype=torch.float16) if k == 'pixel_value' else v.to(device) for k, v in inputs.items()}
        
        generated_ids = model.generate(**inputs, max_new_tokens=50)
        batch_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
        
        generated_texts.extend([t.strip() for t in batch_texts])

    df = pd.DataFrame(data={'image_path': image_paths, 'text': generated_texts})
    df.head()
    df.to_csv("generated_captions.csv", index=False)

In [9]:
generated_texts

[]

In [10]:
batch_size = 4
generated_texts = []

for i in tqdm(range(0, len(image_paths), batch_size)):
    batch_paths = image_paths[i:i+batch_size]
    images = [Image.open(p) for p in batch_paths]
    
    inputs = processor(images=images, text=['Base on state of the product. The product in the picture can be described as:'] * batch_size, return_tensors="pt")
    
    inputs = {k: v.to(device, dtype=torch.float16) if k == 'pixel_value' else v.to(device) for k, v in inputs.items()}
    
    generated_ids = model.generate(**inputs, max_new_tokens=150)
    batch_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
    
    generated_texts.extend([t.strip() for t in batch_texts])
    break

  0%|          | 0/68 [00:00<?, ?it/s]

In [ ]:
generated_texts

['a chair',
 'a) a) a) a) b) c) d) e) f) g) h) i) j) k) l) m) n) o) p) q) r) s) t) u) v) w) x) y) z) z) z) z) z) z) z) z) z) z) z) z) z) z) z) z) z) z) z) z) z) z)',
 'a) new',
 'a new product']

: 

In [ ]:
batch_size = 4
generated_texts = []

for i in tqdm(range(0, len(image_paths), batch_size)):
    batch_paths = image_paths[i:i+batch_size]
    images = [Image.open(p) for p in batch_paths]
    
    inputs = processor(images=images, text=['Question: Is it broken? Answer:'] * batch_size, return_tensors="pt")
    inputs = {k: v.to(device, dtype=torch.float16) if k == 'pixel_value' else v.to(device) for k, v in inputs.items()}
    
    generated_ids = model.generate(**inputs, max_new_tokens=50)
    batch_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
    
    generated_texts.extend([t.strip() for t in batch_texts])
    break

  0%|          | 0/68 [00:00<?, ?it/s]

In [ ]:
generated_texts 

['no', 'no', 'no', 'no']

In [ ]:
batch_size = 4
generated_texts = []

for i in tqdm(range(0, len(image_paths), batch_size)):
    batch_paths = image_paths[i:i+batch_size]
    images = [Image.open(p) for p in batch_paths]
    
    inputs = processor(images=images, text=['Question: Was it fixed? Answer:'] * batch_size, return_tensors="pt")
    inputs = {k: v.to(device, dtype=torch.float16) if k == 'pixel_value' else v.to(device) for k, v in inputs.items()}
    
    generated_ids = model.generate(**inputs, max_new_tokens=50)
    batch_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
    
    generated_texts.extend([t.strip() for t in batch_texts])
    break

  0%|          | 0/68 [00:00<?, ?it/s]

In [ ]:
generated_texts

['yes', 'yes', 'yes', 'yes']

In [ ]:
batch_size = 4
generated_texts = []

for i in tqdm(range(0, len(image_paths), batch_size)):
    batch_paths = image_paths[i:i+batch_size]
    images = [Image.open(p) for p in batch_paths]
    
    inputs = processor(images=images, text=['Question: Is it new? Answer:'] * batch_size, return_tensors="pt")
    inputs = {k: v.to(device, dtype=torch.float16) if k == 'pixel_value' else v.to(device) for k, v in inputs.items()}
    
    generated_ids = model.generate(**inputs, max_new_tokens=50)
    batch_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
    
    generated_texts.extend([t.strip() for t in batch_texts])
    break

  0%|          | 0/68 [00:00<?, ?it/s]

In [ ]:
generated_texts 

['no', 'yes', 'yes', 'yes']